# 不同类型运行的跟踪

### 运行类型

LangSmith 支持多种不同类型的运行 - 您可以在 @traceable 装饰器中指定运行的类型。运行类型包括：

- LLM：调用大语言模型
- Retriever：从数据库或其他来源检索文档
- Tool：通过函数调用执行操作
- Chain：默认类型；将多个运行组合成一个更大的流程
- Prompt：构建提示词以供大语言模型使用
- Parser：提取结构化数据

### 设置

In [ ]:
# 您可以在代码中直接设置！
import os
os.environ["OPENAI_API_KEY"] = ""
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"

In [ ]:
# 或者您可以使用 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

### 聊天模型的 LLM 运行

LangSmith 为 LLM 跟踪提供特殊的渲染和处理。为了充分利用这一功能，您必须以特定格式记录您的 LLM 跟踪。

对于聊天式模型，输入必须是 OpenAI 兼容格式的消息列表，表示为 Python 字典或 TypeScript 对象。每条消息必须包含 role 和 content 键。

输出接受以下任一格式：

- 包含 choices 键的字典/对象，其值为字典/对象列表。每个字典/对象必须包含 message 键，映射到具有 role 和 content 键的消息对象。
- 包含 message 键的字典/对象，其值为具有 role 和 content 键的消息对象。
- 包含两个元素的元组/数组，第一个元素是角色，第二个元素是内容。
- 包含 role 和 content 键的字典/对象。

函数的输入参数应命名为 messages。

您还可以提供以下元数据字段来帮助 LangSmith 识别模型并计算成本。如果使用 LangChain 或 OpenAI 包装器，这些字段将自动正确填充。
- ls_provider：模型提供商，例如 "openai"、"anthropic" 等。
- ls_model_name：模型名称，例如 "gpt-4o-mini"、"claude-3-opus-20240307" 等。

In [ ]:
from langsmith import traceable

inputs = [
  {"role": "system", "content": "您是一个乐于助人的助手。"},
  {"role": "user", "content": "我想预订一张两人桌。"},
]

output = {
  "choices": [
      {
          "message": {
              "role": "assistant",
              "content": "当然，您想预订几点的桌子？"
          }
      }
  ]
}

# 也可以使用以下格式之一：
# output = {
#     "message": {
#         "role": "assistant",
#         "content": "当然，您想预订几点的桌子？"
#     }
# }
#
# output = {
#     "role": "assistant",
#     "content": "当然，您想预订几点的桌子？"
# }
#
# output = ["assistant", "当然，您想预订几点的桌子？"]

@traceable(
  # TODO: 添加 run_type="llm" 以及 ls_provider 和 ls_model_name 的元数据
)
def chat_model(messages: list):
  return output

chat_model(inputs)

### 处理流式 LLM 运行

对于流式输出，您可以将输出"归约"为与非流式版本相同的格式。目前仅在 Python 中支持此功能。

In [ ]:
def _reduce_chunks(chunks: list):
    all_text = "".join([chunk["choices"][0]["message"]["content"] for chunk in chunks])
    return {"choices": [{"message": {"content": all_text, "role": "assistant"}}]}

@traceable(
    run_type="llm",
    metadata={"ls_provider": "my_provider", "ls_model_name": "my_model"},
    # TODO: 添加 reduce_fn
)
def my_streaming_chat_model(messages: list):
    for chunk in ["您好，" + messages[1]["content"]]:
        yield {
            "choices": [
                {
                    "message": {
                        "content": chunk,
                        "role": "assistant",
                    }
                }
            ]
        }

list(
    my_streaming_chat_model(
        [
            {"role": "system", "content": "您是一个乐于助人的助手。请向用户问好。"},
            {"role": "user", "content": "鹦鹉波利"},
        ],
    )
)

### 检索器运行 + 文档

许多 LLM 应用程序需要从向量数据库、知识图谱或其他类型的索引中查找文档。检索器跟踪是记录检索器检索到的文档的一种方式。LangSmith 为跟踪中的检索步骤提供特殊渲染，使理解和诊断检索问题更容易。为了让检索步骤正确渲染，需要执行几个小步骤。

1. 使用 run_type="retriever" 注释检索器步骤。
2. 从检索器步骤返回 Python 字典或 TypeScript 对象的列表。每个字典应包含以下键：
    - page_content：文档的文本。
    - type：这应该始终为 "Document"。
    - metadata：包含有关文档元数据的 Python 字典或 TypeScript 对象。此元数据将在跟踪中显示。

In [ ]:
from langsmith import traceable

def _convert_docs(results):
  return [
      {
          "page_content": r,
          "type": "Document", # 这是错误的格式！键应该是 type
          "metadata": {"foo": "bar"}
      }
      for r in results
  ]

@traceable(
    # TODO: 添加 run_type="retriever"
)
def retrieve_docs(query):
  # 检索器返回硬编码的虚拟文档。
  # 在生产环境中，这可能是真实的向量数据库或其他文档索引。
  contents = ["文档内容 1", "文档内容 2", "文档内容 3"]
  return _convert_docs(contents)

retrieve_docs("用户查询")

### 工具调用

LangSmith 对模型进行的工具调用具有自定义渲染，以便清楚地显示何时使用了提供的工具。

In [ ]:
from langsmith import traceable
from openai import OpenAI
from typing import List, Optional
import json

openai_client = OpenAI()

@traceable(
  # TODO: 添加 run_type="tool"
)
def get_current_temperature(location: str, unit: str):
    return 65 if unit == "Fahrenheit" else 17

@traceable(run_type="llm")
def call_openai(
    messages: List[dict], tools: Optional[List[dict]]
) -> str:
  return openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0,
    tools=tools
  )

@traceable(run_type="chain")
def ask_about_the_weather(inputs, tools):
  response = call_openai(inputs, tools)
  tool_call_args = json.loads(response.choices[0].message.tool_calls[0].function.arguments)
  location = tool_call_args["location"]
  unit = tool_call_args["unit"]
  tool_response_message = {
    "role": "tool",
    "content": json.dumps({
        "location": location,
        "unit": unit,
        "temperature": get_current_temperature(location, unit),
    }),
    "tool_call_id": response.choices[0].message.tool_calls[0].id
  }
  inputs.append(response.choices[0].message)
  inputs.append(tool_response_message)
  output = call_openai(inputs, None)
  return output

tools = [
    {
      "type": "function",
      "function": {
        "name": "get_current_temperature",
        "description": "获取特定位置的当前温度",
        "parameters": {
          "type": "object",
          "properties": {
            "location": {
              "type": "string",
              "description": "城市和州，例如 旧金山，加利福尼亚"
            },
            "unit": {
              "type": "string",
              "enum": ["摄氏度", "华氏度"],
              "description": "要使用的温度单位。从用户的位置推断这一点。"
            }
          },
          "required": ["location", "unit"]
        }
      }
    }
]
inputs = [
  {"role": "system", "content": "您是一个乐于助人的助手。"},
  {"role": "user", "content": "纽约市今天的天气如何？"},
]

ask_about_the_weather(inputs, tools)